# RExA — Training Metrics & Model Comparison

**Project:** Explainable Reasoning Analysis of Descriptive Answers (RExA)

This notebook reports **Accuracy, Precision, Recall, F1** for Core RExA modules and compares them with published descriptive-answer systems (including DAES — IEEE Access 2024).

> **Viva note:** Literature numbers come from other datasets/tasks. Use them as contextual benchmarks. RExA’s contribution is **explainable reasoning structure**, not only a single score.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('../..').resolve()
RESULTS = ROOT / 'data' / 'baselines' / 'large_results.json'
COMPARE = ROOT / 'data' / 'baselines' / 'model_comparison.json'
DISTIL = ROOT / 'ml' / 'checkpoints' / 'distilbert_stars' / 'metrics.json'
FIG_DIR = ROOT / 'docs' / 'figures'
PUB_DIR = ROOT / 'public' / 'evaluation' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PUB_DIR.mkdir(parents=True, exist_ok=True)

results = json.loads(RESULTS.read_text(encoding='utf-8'))
compare = json.loads(COMPARE.read_text(encoding='utf-8'))
print('Loaded results from', RESULTS)
print('Corpus:', results['corpus'])

## 1. Core RExA module metrics (Accuracy / Precision / Recall / F1)

Macro averages from the large ASAP test set (`large_results.json`).

In [ ]:
rows = []
for m in compare['rexa_modules']:
    rows.append({
        'Model': m['model'],
        'Accuracy': round(m['accuracy'] * 100, 2),
        'Precision': round(m['precision'] * 100, 2),
        'Recall': round(m['recall'] * 100, 2),
        'F1-score': round(m['f1'] * 100, 2),
    })

module_df = pd.DataFrame(rows)
display(module_df)
print('\n* Support/Contradiction 100% is vs silver heuristic labels — disclose in viva.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(module_df))
w = 0.2
ax.bar(x - 1.5*w, module_df['Accuracy'], w, label='Accuracy', color='#3b82f6')
ax.bar(x - 0.5*w, module_df['Precision'], w, label='Precision', color='#10b981')
ax.bar(x + 0.5*w, module_df['Recall'], w, label='Recall', color='#f59e0b')
ax.bar(x + 1.5*w, module_df['F1-score'], w, label='F1-score', color='#8b5cf6')
ax.set_xticks(x)
ax.set_xticklabels(['Sentence Roles', 'Concept Coverage', 'Support/Contr.*'], rotation=15)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 115)
ax.set_title('RExA Core modules — Accuracy, Precision, Recall, F1')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
for folder in (FIG_DIR, PUB_DIR):
    fig.savefig(folder / '09_rexa_module_clf_metrics.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved 09_rexa_module_clf_metrics.png')

## 2. Per-class metrics — Sentence Roles (Obj 1)

In [ ]:
report = results['modules']['sentence_roles']['classification_report']
role_rows = []
for role in ['Claim', 'Evidence', 'Explanation', 'Conclusion', 'Other']:
    r = report[role]
    role_rows.append({
        'Role': role,
        'Precision': round(r['precision'] * 100, 2),
        'Recall': round(r['recall'] * 100, 2),
        'F1-score': round(r['f1-score'] * 100, 2),
        'Support': int(r['support']),
    })
role_df = pd.DataFrame(role_rows)
display(role_df)
print('Overall role accuracy: {:.2f}%'.format(results['modules']['sentence_roles']['accuracy'] * 100))
print('Macro-F1: {:.4f}'.format(results['modules']['sentence_roles']['macro_f1']))

## 3. Literature comparison (DAES & others vs RExA)

Published Acc/P/R/F1 from related descriptive-answer papers, plus **RExA Sentence Roles** (our proposed classification module).

In [ ]:
lit_rows = []
for item in compare['literature']:
    lit_rows.append({
        'Model': item['model'],
        'Accuracy %': None if item['accuracy'] is None else round(item['accuracy'] * 100, 2),
        'Precision %': None if item.get('precision') is None else round(item['precision'] * 100, 2),
        'Recall %': None if item.get('recall') is None else round(item['recall'] * 100, 2),
        'F1 %': None if item.get('f1') is None else round(item['f1'] * 100, 2),
        'Focus': item.get('focus', ''),
    })
lit_df = pd.DataFrame(lit_rows)
display(lit_df)
print('\nKey citation: DAES — DOI 10.1109/ACCESS.2024.3417706')

In [ ]:
# Compare models that publish full Acc/P/R/F1
plot_items = [x for x in compare['literature'] if x.get('f1') is not None]
labels = [x['model'].replace(' (ours)', '\n(ours)') for x in plot_items]
metrics = ['accuracy', 'precision', 'recall', 'f1']
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1']
colors = ['#3b82f6', '#10b981', '#f59e0b', '#8b5cf6']

x = np.arange(len(plot_items))
w = 0.2
fig, ax = plt.subplots(figsize=(11, 5.5))
for i, (key, name, color) in enumerate(zip(metrics, metric_names, colors)):
    vals = [plot_items[j][key] * 100 for j in range(len(plot_items))]
    bars = ax.bar(x + (i - 1.5) * w, vals, w, label=name, color=color)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.8, f'{v:.1f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Score (%)')
ax.set_ylim(80, 105)
ax.set_title('Model comparison — Acc / Precision / Recall / F1\n(DAES & hybrids vs RExA Sentence Roles)')
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.12))
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
for folder in (FIG_DIR, PUB_DIR):
    fig.savefig(folder / '10_literature_model_comparison.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved 10_literature_model_comparison.png')

## 4. How RExA differs (for viva)

| Aspect | DAES / typical scoring papers | RExA (ours) |
|--------|-------------------------------|-------------|
| Main goal | Predict a grade / correctness | Analyze **reasoning structure** |
| Signals | Topics, T5 QA, SBERT similarity | Roles, coverage, support, depth |
| Output | Mostly a score | Score + **explainable** roles/links/feedback |
| Metrics shown | Acc/P/R/F1 on scoring labels | Acc/P/R/F1 on **roles** + MAE for stars |

RExA Sentence Roles reaches **~95.9% accuracy** and **~94.5% macro-F1**, competitive with DAES’s published Acc/F1, while adding explainable reasoning analysis.

## 5. Star-scoring comparison (regression-style metrics)

Star models are better judged with **MAE / within-1-star / Spearman** (not exact Acc alone).

In [ ]:
score_rows = []
for s in compare['rexa_scoring']:
    score_rows.append({
        'Model': s['model'],
        'MAE (↓)': round(s['mae'], 3),
        'Within-1-star % (↑)': round(s['within_one_star'] * 100, 2),
        'Exact Acc %': round(s['exact_accuracy'] * 100, 2),
        'Spearman ρ': round(s['spearman'], 3),
    })
score_df = pd.DataFrame(score_rows)
display(score_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(score_df['Model'], score_df['MAE (↓)'], color=['#f87171', '#34d399', '#60a5fa'])
axes[0].set_title('Star MAE (lower is better)')
axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(score_df['Model'], score_df['Within-1-star % (↑)'], color=['#f87171', '#34d399', '#60a5fa'])
axes[1].set_title('Within-1-star accuracy % (higher is better)')
axes[1].tick_params(axis='x', rotation=20)
fig.suptitle('Star scoring: Keyword vs RExA Core vs DistilBERT (comparative)', fontweight='bold')
fig.tight_layout()
for folder in (FIG_DIR, PUB_DIR):
    fig.savefig(folder / '11_star_scoring_comparison.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved 11_star_scoring_comparison.png')

## 6. Export summary for the app / thesis

Writes a compact JSON used by the Evaluation page.

In [ ]:
out = {
    'rexa_clf_table': module_df.to_dict(orient='records'),
    'literature_table': lit_df.fillna('—').to_dict(orient='records'),
    'star_table': score_df.to_dict(orient='records'),
    'figures': [
        '09_rexa_module_clf_metrics.png',
        '10_literature_model_comparison.png',
        '11_star_scoring_comparison.png',
    ],
}
out_path = ROOT / 'public' / 'evaluation' / 'comparison_tables.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(out, indent=2), encoding='utf-8')
print('Wrote', out_path)